In [1]:
!date

Thu Sep 17 23:27:47 PDT 2026


In [2]:
import glob
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor

In [3]:
projdir = '/u/project/cluo/terencew/claude/project_ideas/asm_lr_hprc2'
outdir = f'{projdir}/csv/figure_s1'
!mkdir -p {outdir}

### per-haplotype: depth, CpGs, global mCG, mapping rate (B01a summaries)

In [4]:
hap = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{projdir}/results/meth_bins/per_hap/*.summary.tsv')])
hap = hap.rename(columns={'mean_depth': 'cpg_depth_mean', 'median_depth': 'cpg_depth_median',
                          'n_cpg_native': 'n_cpg_assembly', 'n_cpg_hg38': 'n_cpg_hg38',
                          'global_meth': 'mcg_percpg', 'global_wmeth': 'mcg_callweighted'})

In [5]:
hap.head()

,sample,hap,n_cpg_assembly,mcg_percpg,mcg_callweighted,cpg_depth_mean,cpg_depth_median,pmd_native_bp,frac_cpg_in_pmd_native,meth_in_pmd_native,meth_out_pmd_native,n_cpg_hg38,map_rate,pmd_hg38_bp,n_pmd_hg38
0,HG00097,1,32408753,0.517675,0.639523,25.394918,25.0,1552234578,0.403549,0.563313,0.725688,26498393,0.817631,1349212452,794
0,HG00097,2,32023247,0.523907,0.634447,25.418936,25.0,1549609087,0.403253,0.561277,0.723950,26461936,0.826335,1340601981,730
0,HG00099,1,32420616,0.517486,0.684083,27.427522,27.0,1317374744,0.334136,0.617930,0.745449,26448515,0.815793,1170730688,597
0,HG00099,2,32286272,0.519639,0.678444,27.470538,27.0,1328438984,0.337463,0.617382,0.743672,26468116,0.819795,1133392036,558
0,HG00126,1,31775871,0.527986,0.668717,33.891916,33.0,1336041742,0.365254,0.621466,0.739335,26469735,0.833014,1191635486,590


In [6]:
hap.shape

(402, 15)

### molecule-level QC (Q01a, chr20): reads, spans, het sites per molecule, het-CpG linkages

In [7]:
mol = pd.concat([pd.read_csv(f, sep='\t') for f in glob.glob(f'{projdir}/results/qc/data/molecule_qc/*_chr20.tsv')])
print(len(mol), 'haplotype rows,', mol['sample'].nunique(), 'donors')
mol.head()

402 haplotype rows, 201 donors


,sample,hap,chrom,n_reads,median_span_kb,mean_span_kb,n_het_sites,median_het_spacing_bp,mean_het_per_read,median_het_per_read,frac_reads_ge1_het,frac_reads_ge2_het,mean_cpg_per_read,median_cpg_per_read,het_cpg_linkages,linkages_per_read,mean_read_mcg
0,HG00097,1,chr20,29889,34.6820,53.242527,76213,239.0,59.237612,27.0,0.945967,0.900867,611.867811,360.0,2265324928,75791.258590,0.611921
1,HG00097,2,chr20,30235,35.8360,53.857178,72401,277.0,57.024045,26.0,0.933587,0.886621,621.058376,372.0,2138691318,70735.614950,0.610551
0,HG00099,1,chr20,40877,27.3090,43.648127,89067,191.5,45.912763,21.0,0.942853,0.888690,500.949703,284.0,2120904119,51885.023828,0.659992
1,HG00099,2,chr20,41510,27.7675,44.319352,84167,221.0,46.752180,21.0,0.944568,0.890821,509.658444,289.0,2214321093,53344.280728,0.658557
0,HG00126,1,chr20,30524,54.8790,70.542237,89213,187.0,79.237158,43.0,0.938606,0.902339,816.545800,565.0,3466730178,113573.914887,0.642976


### covariates: chemistry / read N50 (HPRC2 S6), superpopulation, ancestry PCs (1000G PCA)

In [8]:
seq = pd.read_csv(f'{projdir}/results/qc/data/supp_seq_qc.csv')[
    ['sample_id', 'sequencing_chemistry_ont', 'read_N50_ont', 'coverage_ont']].rename(columns={'sample_id': 'sample'})
man = pd.read_csv(f'{projdir}/tsv/meta/hprc2_sample_manifest.tsv', sep='\t')[
    ['sample_id', 'population', 'superpopulation']].rename(columns={'sample_id': 'sample'})
pca = pd.read_csv('/u/project/cluo/terencew/claude/reference/1000G/pca/asm_lr_hprc2/pca_result.eigenvec',
                  sep=r'\s+', header=None)
pca = pca.iloc[:, 1:6]
pca.columns = ['sample', 'PC1', 'PC2', 'PC3', 'PC4']

In [9]:
pca.head()

,sample,PC1,PC2,PC3,PC4
0,HG00097,-0.033251,0.107533,0.030002,0.083610
1,HG00099,-0.038324,0.102197,0.022336,0.079081
2,HG00126,-0.035119,0.104478,0.028886,0.081567
3,HG00128,-0.034674,0.109887,0.027099,0.083104
4,HG00133,-0.035010,0.108044,0.031990,0.078515


### het SNPs: genome-wide counts (H01) and read-filter retention (C02a logs, genome-wide)

In [10]:
### genome-wide het SNVs per donor from the cohort VCF (bcftools stats PSC nHets); QC05 was a 10-donor pilot
het = pd.read_csv(f'{projdir}/results/qc/data/het_snp_counts_per_donor_gw.tsv', sep='\t').rename(
    columns={'n_het_snps': 'n_het_snps_genome'})[['sample', 'n_het_snps_genome']]
ret = pd.read_csv(f'{projdir}/results/asm/genome/donor_summary.tsv', sep='\t')[
    ['sample', 'frac_reads_het', 'frac_reads_kept', 'frac_asm', 'n_tested', 'n_asm', 'median_reads']]

### methylome state and ASM calibration per donor

In [11]:
dm = pd.read_csv(f'{projdir}/results/qc/data/qc13/donor_pmd_expansion_metrics.tsv', sep='\t')[
    ['sample', 'depth_constitutive', 'mcg_constitutive', 'mcg_never', 'own_pmd_bin_frac']]
lam = pd.read_csv(f'{projdir}/results/qc/data/asm_null_vs_real_chr20.tsv', sep='\t')[
    ['sample', 'lambda_gc', 'lambda_null', 'read_sd']] if len(glob.glob(f'{projdir}/results/qc/data/asm_null_vs_real_chr20.tsv')) else pd.DataFrame(columns=['sample'])
xist = pd.read_csv(f'{projdir}/results/qc/data/xist_promoter_skew.tsv', sep='\t')[
    ['sample', 'skew']].rename(columns={'skew': 'xist_skew'})

### assemble: per-haplotype table and per-donor table

In [12]:
hapq = hap.merge(mol.drop(columns=['chrom']), on=['sample', 'hap'], how='left').merge(seq, on='sample', how='left').merge(
    man, on='sample', how='left').merge(pca, on='sample', how='left')
hapq['pmd_gb'] = hapq['pmd_native_bp'] / 1e9
hapq['pmd_contrast'] = hapq['meth_out_pmd_native'] - hapq['meth_in_pmd_native']

In [13]:
hapq.head()

,sample,hap,n_cpg_assembly,mcg_percpg,mcg_callweighted,cpg_depth_mean,cpg_depth_median,pmd_native_bp,frac_cpg_in_pmd_native,meth_in_pmd_native,...,read_N50_ont,coverage_ont,population,superpopulation,PC1,PC2,PC3,PC4,pmd_gb,pmd_contrast
0,HG00097,1,32408753,0.517675,0.639523,25.394918,25.0,1552234578,0.403549,0.563313,...,89210.35,57.09,GBR,EUR,-0.033251,0.107533,0.030002,0.083610,1.552235,0.162375
1,HG00097,2,32023247,0.523907,0.634447,25.418936,25.0,1549609087,0.403253,0.561277,...,89210.35,57.09,GBR,EUR,-0.033251,0.107533,0.030002,0.083610,1.549609,0.162673
2,HG00099,1,32420616,0.517486,0.684083,27.427522,27.0,1317374744,0.334136,0.617930,...,72139.93,58.78,GBR,EUR,-0.038324,0.102197,0.022336,0.079081,1.317375,0.127519
3,HG00099,2,32286272,0.519639,0.678444,27.470538,27.0,1328438984,0.337463,0.617382,...,72139.93,58.78,GBR,EUR,-0.038324,0.102197,0.022336,0.079081,1.328439,0.126290
4,HG00126,1,31775871,0.527986,0.668717,33.891916,33.0,1336041742,0.365254,0.621466,...,107245.82,70.94,GBR,EUR,-0.035119,0.104478,0.028886,0.081567,1.336042,0.117869


In [14]:
hapq.shape

(402, 40)

In [15]:
wide = hapq.pivot_table(index='sample', columns='hap',
                        values=['cpg_depth_mean', 'n_cpg_hg38', 'mcg_callweighted', 'map_rate',
                                'n_reads', 'median_span_kb', 'mean_het_per_read', 'frac_reads_ge1_het',
                                'het_cpg_linkages', 'linkages_per_read', 'n_het_sites', 'median_het_spacing_bp'])
wide.columns = [f'{a}_hap{int(b)}' for a, b in wide.columns]
donor = wide.reset_index().merge(seq, on='sample', how='left').merge(man, on='sample', how='left').merge(
    pca, on='sample', how='left').merge(het, on='sample', how='left').merge(ret, on='sample', how='left').merge(
    dm, on='sample', how='left').merge(xist, on='sample', how='left')
if len(lam):
    donor = donor.merge(lam, on='sample', how='left')
donor['mcg_hap_diff'] = (donor['mcg_callweighted_hap1'] - donor['mcg_callweighted_hap2'])
donor['cpg_depth_mean'] = donor[['cpg_depth_mean_hap1', 'cpg_depth_mean_hap2']].mean(axis=1)
donor['het_cpg_linkages_total'] = donor[['het_cpg_linkages_hap1', 'het_cpg_linkages_hap2']].sum(axis=1)

In [16]:
donor.head()

,sample,cpg_depth_mean_hap1,cpg_depth_mean_hap2,frac_reads_ge1_het_hap1,frac_reads_ge1_het_hap2,het_cpg_linkages_hap1,het_cpg_linkages_hap2,linkages_per_read_hap1,linkages_per_read_hap2,map_rate_hap1,...,mcg_constitutive,mcg_never,own_pmd_bin_frac,xist_skew,lambda_gc,lambda_null,read_sd,mcg_hap_diff,cpg_depth_mean,het_cpg_linkages_total
0,HG00097,25.394918,25.418936,0.945967,0.933587,2265324928,2138691318,75791.258590,70735.614950,0.817631,...,0.532610,0.778194,0.494646,0.471476,0.912492,0.670178,0.152912,0.005077,25.406927,4404016246
1,HG00099,27.427522,27.470538,0.942853,0.944568,2120904119,2214321093,51885.023828,53344.280728,0.815793,...,0.591849,0.789830,0.426773,0.781786,3.309045,0.659745,0.123068,0.005639,27.449030,4335225212
2,HG00126,33.891916,33.572569,0.938606,0.955062,3466730178,3542596315,113573.914887,122458.305334,0.833014,...,0.593786,0.782072,0.441540,NaN,0.962873,0.630849,0.162299,0.000426,33.732242,7009326493
3,HG00128,26.742351,26.842680,0.929223,0.932320,2281471480,2309032863,71735.362847,73300.303578,0.814603,...,0.552431,0.788791,0.508949,0.455455,1.385617,0.658168,0.158349,-0.000425,26.792516,4590504343
4,HG00133,30.973569,31.137178,0.964940,0.960794,3149299203,3060083830,108461.881905,107889.991538,0.838005,...,0.585918,0.787096,0.446916,0.273500,0.994822,0.629912,0.158534,-0.000768,31.055373,6209383033


In [17]:
donor.shape

(201, 52)

### summaries printed for the record

In [18]:
cols = ['cpg_depth_mean', 'n_cpg_hg38_hap1', 'mcg_callweighted_hap1', 'n_het_snps_genome',
        'median_het_spacing_bp_hap1', 'mean_het_per_read_hap1', 'frac_reads_ge1_het_hap1',
        'linkages_per_read_hap1', 'frac_reads_kept', 'mcg_hap_diff']
donor[cols].describe().loc[['mean', 'std', 'min', '50%', 'max']].round(3)

,cpg_depth_mean,n_cpg_hg38_hap1,mcg_callweighted_hap1,n_het_snps_genome,median_het_spacing_bp_hap1,mean_het_per_read_hap1,frac_reads_ge1_het_hap1,linkages_per_read_hap1,frac_reads_kept,mcg_hap_diff
mean,30.183,2.645754e+07,0.662,6769765.000,217.823,59.306,0.943,68746.783,0.945,0.003
std,4.415,2.488857e+04,0.041,756709.031,43.207,13.249,0.034,22763.277,0.023,0.003
min,15.650,2.630582e+07,0.415,6161890.000,104.000,33.927,0.772,30537.999,0.858,-0.004
50%,29.745,2.645871e+07,0.666,6618723.500,213.000,57.040,0.946,64916.434,0.943,0.003
max,41.631,2.650456e+07,0.748,8770733.000,327.000,104.645,0.989,162439.952,0.981,0.013


In [19]:
donor.groupby('superpopulation')[['cpg_depth_mean', 'n_het_snps_genome', 'frac_reads_ge1_het_hap1', 'frac_reads_kept', 'mcg_callweighted_hap1']].median().round(3)

,cpg_depth_mean,n_het_snps_genome,frac_reads_ge1_het_hap1,frac_reads_kept,mcg_callweighted_hap1
superpopulation,,,,,
AFR,29.163,8770733.0,0.972,0.972,0.659
AMR,30.862,6653060.0,0.937,0.937,0.680
EAS,31.550,6179010.5,0.934,0.931,0.670
EUR,29.745,6384562.5,0.940,0.934,0.639
SAS,28.913,6744368.5,0.951,0.947,0.686


In [20]:
donor.groupby('sequencing_chemistry_ont')[['cpg_depth_mean', 'read_N50_ont', 'mcg_callweighted_hap1', 'frac_reads_kept']].median().round(3)

,cpg_depth_mean,read_N50_ont,mcg_callweighted_hap1,frac_reads_kept
sequencing_chemistry_ont,,,,
R1041,30.305,81787.17,0.637,0.939
R941,29.381,80364.98,0.676,0.948


### export for the R figure notebook

In [21]:
hapq.to_csv(f'{outdir}/s1_hap_qc.csv', sep='\t', index=False)
donor.to_csv(f'{outdir}/s1_donor_qc.csv', sep='\t', index=False)
long = hapq.melt(id_vars=['sample', 'hap', 'superpopulation', 'population', 'sequencing_chemistry_ont', 'PC1', 'PC2'],
                 value_vars=['cpg_depth_mean', 'n_cpg_hg38', 'mcg_callweighted', 'map_rate', 'n_reads',
                             'median_span_kb', 'mean_het_per_read', 'frac_reads_ge1_het', 'linkages_per_read',
                             'n_het_sites', 'median_het_spacing_bp', 'pmd_gb', 'pmd_contrast'],
                 var_name='metric', value_name='value')
long.to_csv(f'{outdir}/s1_hap_qc_long.csv', sep='\t', index=False)
print(long.metric.nunique(), 'metrics x', long['sample'].nunique(), 'donors exported')

13 metrics x 201 donors exported


In [22]:
!date

Thu Sep 17 23:27:56 PDT 2026
